In [ ]:
%pip install -U ultralytics

In [ ]:
!nvidia-smi

In [ ]:
import torch

torch.cuda.is_available()

In [ ]:
from ultralytics import solutions

inf = solutions.Inference(
    model="yolo11n.pt",  # you can use any model that Ultralytics supports, e.g., YOLO11, YOLOv10
)

inf.inference()

# Make sure to run the file using command `streamlit run path/to/file.py`

In [ ]:
!yolo solutions inference

!yolo solutions inference model="yolo26n.pt" # use model fine-tuned with Ultralytics Python package!

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)


In [ ]:
### AI Security Servellience as AiSecurityGuard

In [1]:
# import cv2
# from numpy import source

from ultralytics import solutions
from ultralytics.utils.plotting import Annotator

import os
import cv2
import numpy as np
import face_recognition
import pygame

# from ultralytics import solutions
from ultralytics import YOLO
from ultralytics.solutions.config import SolutionConfig
from ultralytics.utils import LOGGER

from ultralytics.solutions.solutions import BaseSolution, SolutionAnnotator, SolutionResults
from ultralytics.utils.plotting import colors

# ========== 🔊 SOUND SETUP ==========
pygame.mixer.init()
ALARM_FILE = "../media_files/Alarm-sound-samples/humordome-security-alert-sound-453297.mp3"
if os.path.exists(ALARM_FILE):
    pygame.mixer.music.load(ALARM_FILE)
else:
    print(f"[WARNING] Alarm file '{ALARM_FILE}' not found.")


# ========== 🧠 KNOWN FACE ENCODING LOADER ==========
KNOWN_FACE_DIR = "../family_members/"
known_face_encodings, known_face_names = [], []

if os.path.exists(KNOWN_FACE_DIR):
    for name in os.listdir(KNOWN_FACE_DIR):
        person_dir = os.path.join(KNOWN_FACE_DIR, name)
        if not os.path.isdir(person_dir):
            continue
        for filename in os.listdir(person_dir):
            path = os.path.join(person_dir, filename)
            try:
                img = face_recognition.load_image_file(path)
                enc = face_recognition.face_encodings(img)
                if enc:
                    known_face_encodings.append(enc[0])
                    known_face_names.append(name)
                    print(f"[INFO] Loaded face for {name} from {filename}")
            except Exception as e:
                print(f"[ERROR] Failed loading {path}: {e}")
else:
    print("[WARNING] No known_faces directory found.")


# ========== 👁️ FACE-RECOGNITION ALARM (REVISED & OPTIMIZED) ==========
class FaceRecognitionAlarmVisionEye(solutions.VisionEye):
    def __init__(self, *args, known_face_encodings=None, known_face_names=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.known_face_encodings = known_face_encodings or []
        self.known_face_names = known_face_names or []
        self.sound_played = False
        # Best practice: Set face recognition tolerance during initialization
        self.face_tolerance = 0.55
        self.vision_point = self.CFG["vision_point"]
        self.records = self.CFG.get("records", 1)
        # self.show = self.CFG.get("show", True)

    def play_sound(self):
        """Plays the alarm sound if it's not already playing."""
        if not self.sound_played:
            if pygame.mixer.get_init() and not pygame.mixer.music.get_busy():
                pygame.mixer.music.play()
                self.sound_played = True
                LOGGER.info("🚨 Alarm Triggered: Unknown person count reached threshold.")

    def reset_sound(self):
        """Stops the alarm sound and resets the state."""
        if self.sound_played:
            if pygame.mixer.get_init():
                pygame.mixer.music.stop()
            self.sound_played = False
            LOGGER.info("🟢 Alarm Reset: Area clear.")

    def __call__(self, im0):
        """
        Processes a single frame for person detection and face recognition.
        This implementation follows best practices for accuracy and performance.
        """
        # 1. Get person detections from the base class
        self.extract_tracks(im0)
        annotator = SolutionAnnotator(im0, line_width=self.line_width)

        unknown_person_count = 0

        # 2. Optimize by finding all faces in the frame at once (on a smaller version)
        # This is much faster than processing crops for each person.
        h, w, _ = im0.shape
        small_frame = cv2.resize(im0, (0, 0), fx=0.25, fy=0.25)
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

        # 3. Iterate through detected PERSONS from YOLO
        for box, conf, cls, t_id in zip(self.boxes, self.confs, self.clss, self.track_ids):
            if int(cls) == 0:  # Skip if not a person
                name = "Unknown"
                is_known = False

                # 4. Associate faces with person boxes
                # Check if any detected face is inside this person's bounding box
                person_box_left, person_box_top, person_box_right, person_box_bottom = map(int, box)

                for (face_top, face_right, face_bottom, face_left), face_encoding in zip(
                    face_locations, face_encodings
                ):
                    # Scale face locations back to original image size
                    face_top *= 4
                    face_right *= 4
                    face_bottom *= 4
                    face_left *= 4

                    # Check if the center of the face is inside the person's box
                    face_center_x = (face_left + face_right) // 2
                    face_center_y = (face_top + face_bottom) // 2

                    if (
                        person_box_left <= face_center_x <= person_box_right
                        and person_box_top <= face_center_y <= person_box_bottom
                    ):
                        # 5. Use robust face matching for the associated face
                        if self.known_face_encodings:
                            face_distances = face_recognition.face_distance(self.known_face_encodings, face_encoding)
                            best_match_index = np.argmin(face_distances)

                            if face_distances[best_match_index] < self.face_tolerance:
                                name = self.known_face_names[best_match_index]
                                is_known = True

                        # Once a face is matched to this person, stop checking other faces
                        break

                # 6. Update counter and draw labels
                if not is_known:
                    unknown_person_count += 1
                    color = (0, 0, 255)  # Red for Unknown
                    # label = f"Unknown ({conf:.2f})"
                    label = f"Unknown"
                else:
                    color = (0, 255, 0)  # Green for Known
                    label = f"{name}"
                    # label = f"{name} ({conf:.2f})"

                # annotator.box_label(box, label, color=color)

                # annotator.visioneye(box, self.vision_point)
                # build base label from the existing adjust_box_label()
                base_label = self.adjust_box_label(int(cls), float(conf) if conf is not None else 0.0, t_id)

                # custom label for 'person' class (COCO id 0). Use CFG override if provided.
                if int(cls) == 0:
                    prefix = str(self.CFG.get("person_label_prefix", label))
                    custom_label = f"{prefix}:"
                    # if base_label exists, concat both for full display
                    final_label = f"{custom_label} {base_label}" if base_label else custom_label
                else:
                    final_label = base_label

                # draw final label and vision eye mapping
                annotator.box_label(box, label=final_label, color=colors(int(t_id), True))
            else:
                # For non-person classes, use default labeling
                annotator.box_label(box, label=self.adjust_box_label(cls, conf, t_id), color=colors(int(t_id), True))

            annotator.visioneye(box, self.vision_point)

        # 7. Trigger alarm based on the COUNT of unknown people and the 'records' threshold
        if unknown_person_count >= self.records:
            self.play_sound()
        else:
            self.reset_sound()

        plot_im = annotator.result()
        self.display_output(plot_im)

        # Display track count on the frame
        total_tracks = len(getattr(self, "track_ids", []))
        cv2.putText(plot_im, f"Tracks: {total_tracks}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        return SolutionResults(plot_im=plot_im, total_tracks=len(self.track_ids))


if __name__ == "__main__":
    # cap = cv2.VideoCapture(0)
    # cap = cv2.VideoCapture("../media_files/istockphoto-2205327227-640_adpp_is.mp4")
    cap = cv2.VideoCapture("../media_files/WIN_20260227_22_00_29_Pro.mp4")
    # cap = cv2.VideoCapture("media_files/person/ruhama/VID_20251122_142652.mp4")
    # cap = cv2.VideoCapture("../media_files/istockphoto-2205327227-640_adpp_is.mp4")
    # cap = cv2.VideoCapture("../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4")
    # cap = cv2.VideoCapture("../media_files/Logi C270 HD WebCam 2025-10-29 22-29-17.mp4")
    # cap = cv2.VideoCapture("../media_files/ruhama.mp4")
    # cap = cv2.VideoCapture("../media_files/ruhama.mp4")
    # assert cap.isOpened(), "Error reading video file"

    # Video writer
    w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
    video_writer = cv2.VideoWriter("visioneye_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    # Initialize vision eye object
    visioneyeInterface = FaceRecognitionAlarmVisionEye(
        show=True,  # display the output
        model="yolo26m.pt",  # use any model that Ultralytics support, i.e, YOLOv10
        # classes=[0, 19],  # generate visioneye view for specific classes
        vision_point=(50, 50, 50),  # the point, where vision will view objects and draw tracks
        known_face_encodings=known_face_encodings,
        known_face_names=known_face_names,
        records=1,  # number of unknown persons to trigger alqqarm
        conf=0.5,
        iou=0.7,
        # verbose=True,
        # show_labels=True,
    )
    # print(visioneyeInterface)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = visioneyeInterface(im0)

    print(results)  # access the output

    # video_writer.write(results.plot_im)  # write the video file

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.8)
Hello from the pygame community. https://www.pygame.org/contribute.html
[INFO] Loaded face for Dulal from WIN_20260210_23_16_20_Pro.jpg
[INFO] Loaded face for Hasan from WIN_20260208_11_14_05_Pro_edited.jpg
[INFO] Loaded face for robin from robin2.jpg
[INFO] Loaded face for robin from WIN_20251215_12_19_35_Pro.jpg
[INFO] Loaded face for sana from Screenshot 2026-01-22 161812.jpg
[INFO] Loaded face for sana from Screenshot 2026-02-08 004947_edited.png
Ultralytics Solutions:  {'source': None, 'model': 'yolo26m.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (50, 50, 50), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 1, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show

error: OpenCV(4.11.0) :-1: error: (-5:Bad argument) in function 'circle'
> Overload resolution failed:
>  - Can't parse 'center'. Expected sequence length 2, got 3
>  - Can't parse 'center'. Expected sequence length 2, got 3


In [ ]:
from ppadb.client import Client as AdbClient

# Connect to ADB server
client = AdbClient(host="127.0.0.1", port=5037)
devices = client.devices()
device = devices[0]

# Make the call
phone_number = "1234567890"
device.shell(f"am start -a android.intent.action.CALL -d tel:{phone_number}")


In [ ]:
# from numpy import source

from ultralytics import solutions
from ultralytics.utils.plotting import Annotator

import os
import cv2

import numpy as np
import face_recognition
import pygame

# from ultralytics import solutions
from ultralytics import YOLO
from ultralytics.solutions.config import SolutionConfig
from ultralytics.utils import LOGGER

from ultralytics.solutions.solutions import BaseSolution, SolutionAnnotator, SolutionResults
from ultralytics.utils.plotting import colors

# ========== ⚙️ CONFIGURATION ==========
SAVE_DIR = "person_cropped_face"
SHARPNESS_THRESHOLD = 40  # Minimum Laplacian variance to consider "not blurry"
os.makedirs(SAVE_DIR, exist_ok=True)

# ========== 🔊 SOUND SETUP ==========
pygame.mixer.init()
ALARM_FILE = "../security_alart.mp3"
if os.path.exists(ALARM_FILE):
    pygame.mixer.music.load(ALARM_FILE)

# ========== 🧠 KNOWN FACE LOADER (Unchanged) ==========
# ... (Keep your existing KNOWN_FACE_DIR loading logic here) ...
KNOWN_FACE_DIR = "../family_members/"
known_face_encodings, known_face_names = [], []
# ... [Your existing loading code] ...


class FaceRecognitionAlarmVisionEye(solutions.VisionEye):
    def __init__(self, *args, known_face_encodings=None, known_face_names=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.known_face_encodings = known_face_encodings or []
        self.known_face_names = known_face_names or []
        self.sound_played = False
        self.face_tolerance = 0.55
        self.vision_point = self.CFG["vision_point"]
        self.records = self.CFG.get("records", 1)

        # New: Tracking best scores to prevent saving thousands of low-quality images
        # Structure: { track_id: highest_quality_score }
        self.best_scores = {}

    def get_quality_score(self, face_img):
        """Calculates a score based on exposure and sharpness."""
        if face_img.size == 0:
            return 0

        # 1. Exposure Score (Target brightness 128)
        yuv = cv2.cvtColor(face_img, cv2.COLOR_BGR2YUV)
        avg_brightness = np.mean(yuv[:, :, 0])
        exposure_score = 255 - abs(avg_brightness - 128)

        # 2. Sharpness Score (Laplacian variance)
        sharpness_score = cv2.Laplacian(face_img, cv2.CV_64F).var()

        # If too blurry, discard
        if sharpness_score < SHARPNESS_THRESHOLD:
            return 0

        return exposure_score + (sharpness_score * 0.5)

    def save_best_crop(self, im0, box, person_id, name, quality):
        """Saves the face crop if it's the best seen so far for this ID."""
        # Only save if quality is significantly better than previous best for this track
        if quality > self.best_scores.get(person_id, 0):
            self.best_scores[person_id] = quality

            # Define folder: person_cropped_face/John_Doe/ or person_cropped_face/ID_5/
            folder_name = name.replace(" ", "_") if name != "Unknown" else f"Unknown_ID_{person_id}"
            path = os.path.join(SAVE_DIR, folder_name)
            os.makedirs(path, exist_ok=True)

            # Crop logic with 20% padding
            x1, y1, x2, y2 = map(int, box)
            h, w, _ = im0.shape
            pw, ph = int((x2 - x1) * 0.2), int((y2 - y1) * 0.2)
            crop = im0[max(0, y1 - ph) : min(h, y2 + ph), max(0, x1 - pw) : min(w, x2 + pw)]

            if crop.size > 0:
                filename = f"best_face_score_{int(quality)}.jpg"
                full_path = os.path.join(path, filename)
                cv2.imwrite(full_path, crop)
                LOGGER.info(f"📸 Saved best face for {folder_name} (Score: {int(quality)})")

    def play_sound(self):
        """Plays the alarm sound if it's not already playing."""
        if not self.sound_played:
            if pygame.mixer.get_init() and not pygame.mixer.music.get_busy():
                pygame.mixer.music.play()
                self.sound_played = True
                LOGGER.info("🚨 Alarm Triggered: Unknown person count reached threshold.")

    def reset_sound(self):
        """Stops the alarm sound and resets the state."""
        if self.sound_played:
            if pygame.mixer.get_init():
                pygame.mixer.music.stop()
            self.sound_played = False
            LOGGER.info("🟢 Alarm Reset: Area clear.")

    def __call__(self, im0):
        self.extract_tracks(im0)
        annotator = SolutionAnnotator(im0, line_width=self.line_width)
        unknown_person_count = 0

        # Optimization: Face detection on a smaller frame
        small_frame = cv2.resize(im0, (0, 0), fx=0.25, fy=0.25)
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

        for box, conf, cls, t_id in zip(self.boxes, self.confs, self.clss, self.track_ids):
            if int(cls) == 0:  # Class 'person'
                name = "Unknown"
                is_known = False

                # Check for face inside person box
                p_x1, p_y1, p_x2, p_y2 = map(int, box)

                for (f_top, f_right, f_bottom, f_left), f_enc in zip(face_locations, face_encodings):
                    # Scale back
                    f_top, f_right, f_bottom, f_left = f_top * 4, f_right * 4, f_bottom * 4, f_left * 4

                    f_cx, f_cy = (f_left + f_right) // 2, (f_top + f_bottom) // 2

                    if p_x1 <= f_cx <= p_x2 and p_y1 <= f_cy <= p_y2:
                        # Identify
                        if self.known_face_encodings:
                            distances = face_recognition.face_distance(self.known_face_encodings, f_enc)
                            if len(distances) > 0:
                                best_idx = np.argmin(distances)
                                if distances[best_idx] < self.face_tolerance:
                                    name = self.known_face_names[best_idx]
                                    is_known = True

                        # --- NEW CROP LOGIC START ---
                        face_roi = im0[f_top:f_bottom, f_left:f_right]
                        quality = self.get_quality_score(face_roi)

                        if quality > 0:
                            # We use t_id (YOLO Track ID) to ensure we save for the right person
                            self.save_best_crop(im0, [f_left, f_top, f_right, f_bottom], t_id, name, quality)
                        # --- NEW CROP LOGIC END ---
                        break

                label = f"{name} ID:{t_id}"
                color = (0, 255, 0) if is_known else (0, 0, 255)
                if not is_known:
                    unknown_person_count += 1

                annotator.box_label(box, label, color=color)
                annotator.visioneye(box, self.vision_point)

        # Alarm Logic
        if unknown_person_count >= self.records:
            self.play_sound()
        else:
            self.reset_sound()

        return SolutionResults(plot_im=annotator.result(), total_tracks=len(self.track_ids))


# Main loop remains basically the same...


if __name__ == "__main__":
    # cap = cv2.VideoCapture(0)
    # cap = cv2.VideoCapture("../media_files/WIN_20251103_14_11_20_Pro.mp4")
    # cap = cv2.VideoCapture("media_files/person/ruhama/VID_20251122_142652.mp4")
    # cap = cv2.VideoCapture("../media_files/istockphoto-2240284006-640_adpp_is.mp4")
    cap = cv2.VideoCapture("../media_files/WIN_20251103_14_11_20_Pro.mp4")
    # cap = cv2.VideoCapture("../media_files/istockphoto-2240284006-640_adpp_is.mp4")
    # cap = cv2.VideoCapture("../media_files/ruhama.mp4")
    # assert cap.isOpened(), "Error reading video file"

    # Video writer
    w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
    video_writer = cv2.VideoWriter("visioneye_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    # Initialize vision eye object
    visioneyeInterface = FaceRecognitionAlarmVisionEye(
        show=True,  # display the output
        model="yolo11n.pt",  # use any model that Ultralytics support, i.e, YOLOv10
        # classes=[0, 19],  # generate visioneye view for specific classes
        vision_point=(50, 50),  # the point, where vision will view objects and draw tracks
        known_face_encodings=known_face_encodings,
        known_face_names=known_face_names,
        records=1,  # number of unknown persons to trigger alarm
        conf=0.5,
        iou=0.7,
        # verbose=True,
        # show_labels=True,
    )


# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = visioneyeInterface(im0)

    print(results)  # access the output

    video_writer.write(results.plot_im)  # write the video file

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()


In [ ]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4")
assert cap.isOpened(), "Error reading video file"

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("security_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

from_email = "abc@gmail.com"  # the sender email address
password = "---- ---- ---- ----"  # 16-digits password generated via: https://myaccount.google.com/apppasswords
to_email = "xyz@gmail.com"  # the receiver email address

# Initialize security alarm object
securityalarm = solutions.SecurityAlarm(
    show=True,  # display the output
    model="yolo26n.pt",  # e.g., yolo26s.pt, yolo26m.pt
    records=1,  # total detections count to send an email
)

# securityalarm.authenticate(from_email, password, to_email)  # authenticate the email server

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = securityalarm(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)  # write the processed frame.

cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows


In [ ]:
import mediapipe as mp
import cv2
import numpy as np
import os
from scipy.spatial import distance as dist

# Configuration
THRESHOLD_MIN = 80
THRESHOLD_MAX = 200
SAVE_DIR = "cropped_faces"
os.makedirs(SAVE_DIR, exist_ok=True)

# Initialize MediaPipe face detector
mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.6)


def get_quality_score(image):
    """Calculates a quality score based on sharpness (Laplacian) and exposure."""
    sharpness = cv2.Laplacian(image, cv2.CV_64F).var()
    yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    brightness = np.mean(yuv[:, :, 0])
    exposure_score = 255 - abs(brightness - 128)
    return (sharpness * exposure_score), brightness


cap = cv2.VideoCapture("../media_files/istockphoto-2240284006-640_adpp_is.mp4")
if not cap.isOpened():
    raise RuntimeError("Cannot open video file.")

# Tracking state
next_id = 0
trackers = {}  # {track_id: centroid}
best_shots = {}  # {track_id: {'score': float, 'crop': image, 'brightness': float}}

print("Processing video to extract the best unique face for each person...")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    ih, iw, _ = frame.shape
    results = face_detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    current_frame_faces = []
    if results.detections:
        for detection in results.detections:
            bbox = detection.location_data.relative_bounding_box
            x, y = max(0, int(bbox.xmin * iw)), max(0, int(bbox.ymin * ih))
            w, h = max(1, int(bbox.width * iw)), max(1, int(bbox.height * ih))
            cx, cy = x + w // 2, y + h // 2
            current_frame_faces.append({"box": (x, y, w, h), "centroid": (cx, cy)})

    # Match detected faces to persistent IDs using Centroid Tracking
    if current_frame_faces:
        if not trackers:
            for f in current_frame_faces:
                trackers[next_id] = f["centroid"]
                next_id += 1
        else:
            ids = list(trackers.keys())
            coords = list(trackers.values())

            for f in current_frame_faces:
                # Find closest existing person
                distances = dist.cdist([f["centroid"]], coords)
                idx = np.argmin(distances)

                if distances[0][idx] < 120:  # Threshold for 'same person'
                    tid = ids[idx]
                    trackers[tid] = f["centroid"]

                    # Evaluate quality for current track
                    x, y, w, h = f["box"]
                    face_roi = frame[y : y + h, x : x + w]
                    if face_roi.size > 0:
                        score, brightness = get_quality_score(face_roi)
                        if THRESHOLD_MIN < brightness < THRESHOLD_MAX:
                            # Update best shot if current quality score is higher
                            if tid not in best_shots or score > best_shots[tid]["score"]:
                                pw, ph = int(w * 0.2), int(h * 0.2)
                                crop = frame[max(0, y - ph) : min(ih, y + h + ph), max(0, x - pw) : min(iw, x + w + pw)]
                                if crop.size > 0:
                                    best_shots[tid] = {"score": score, "crop": crop.copy(), "brightness": brightness}
                else:
                    # New person detected
                    trackers[next_id] = f["centroid"]
                    next_id += 1

    cv2.imshow("Best Face Hunter", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

# Save final unique best faces
print(f"\n✅ Video processed. Found {len(best_shots)} unique persons.")
for tid, data in best_shots.items():
    filename = os.path.join(SAVE_DIR, f"person_{tid}_best.jpg")
    cv2.imwrite(filename, data["crop"])
    print(f"✨ Saved: {filename} (Brightness: {data['brightness']:.1f}, Quality Score: {data['score']:.1f})")

In [ ]:
import mediapipe as mp
import cv2
import numpy as np
import os
from scipy.spatial import distance as dist

# Configuration
THRESHOLD_MIN = 80
THRESHOLD_MAX = 200
SAVE_DIR = "cropped_faces"
os.makedirs(SAVE_DIR, exist_ok=True)

# Initialize MediaPipe face detector with a higher confidence threshold to avoid false positives
mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.7)


def get_quality_score(image):
    """Calculates a quality score based on sharpness (Laplacian) and exposure."""
    sharpness = cv2.Laplacian(image, cv2.CV_64F).var()
    yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    brightness = np.mean(yuv[:, :, 0])
    exposure_score = 255 - abs(brightness - 128)
    return (sharpness * exposure_score), brightness


cap = cv2.VideoCapture("../media_files/istockphoto-2240284006-640_adpp_is.mp4")
if not cap.isOpened():
    raise RuntimeError("Cannot open video file.")

# Tracking state
next_id = 0
trackers = {}  # {track_id: centroid}
best_shots = {}  # {track_id: {'score': float, 'crop': image, 'brightness': float}}

print("Processing video to extract the best unique face for each person...")


def get_sharpness(image):
    return cv2.Laplacian(image, cv2.CV_64F).var()


def get_exposure_score(image):
    yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    avg_brightness = np.mean(yuv[:, :, 0])
    return 255 - abs(avg_brightness - 128)


while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    ih, iw, _ = frame.shape
    results = face_detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    current_frame_faces = []

    if results.detections:
        for detection in results.detections:
            bbox = detection.location_data.relative_bounding_box
            x, y, w, h = int(bbox.xmin * iw), int(bbox.ymin * ih), int(bbox.width * iw), int(bbox.height * ih)
            cx, cy = x + w // 2, y + h // 2
            current_frame_faces.append({"box": (x, y, w, h), "centroid": (cx, cy)})

    # Match detected faces to existing IDs (Simple Centroid Tracking)
    if not trackers:
        for f in current_frame_faces:
            trackers[next_id] = f["centroid"]
            next_id += 1
    else:
        ids = list(trackers.keys())
        coords = list(trackers.values())

        for f in current_frame_faces:
            # Find closest existing ID
            dists = dist.cdist([f["centroid"]], coords)
            idx = np.argmin(dists)

            if dists[0][idx] < 100:  # Threshold for "same person"
                matched_id = ids[idx]
                trackers[matched_id] = f["centroid"]

                # Quality Assessment
                x, y, w, h = f["box"]
                face_roi = frame[max(0, y) : y + h, max(0, x) : x + w]
                if face_roi.size == 0:
                    continue

                sharpness = get_sharpness(face_roi)
                exposure = get_exposure_score(face_roi)
                # Combined Score: Sharpness * Exposure * Size
                total_score = sharpness * exposure * (w * h)

                if matched_id not in best_shots or total_score > best_shots[matched_id]["score"]:
                    # Crop with padding
                    pw, ph = int(w * 0.2), int(h * 0.2)
                    crop = frame[max(0, y - ph) : min(ih, y + h + ph), max(0, x - pw) : min(iw, x + w + pw)]
                    best_shots[matched_id] = {"score": total_score, "crop": crop.copy()}
            else:
                # New person detected
                trackers[next_id] = f["centroid"]
                next_id += 1

    cv2.imshow("Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# cap.release()
# # Save logic remains same...

#     cv2.imshow("Best Face Hunter", frame)
#     if cv2.waitKey(1) & 0xFF == ord("q"):
#         break

cap.release()
cv2.destroyAllWindows()

# Save final unique best faces
print(f"\n✅ Video processed. Found {len(best_shots)} unique persons.")
for tid, data in best_shots.items():
    filename = os.path.join(SAVE_DIR, f"person_{tid}_best.jpg")
    cv2.imwrite(filename, data["crop"])
    print(f"✨ Saved: {filename} (Brightness: {data['brightness']:.1f}, Quality Score: {data['score']:.1f})")


In [ ]:
import mediapipe as mp
import cv2
import numpy as np
import os
from scipy.spatial import distance as dist
from ultralytics import YOLO

# import mediapipe as mp
# Configuration
THRESHOLD_MIN = 80
THRESHOLD_MAX = 200
SAVE_DIR = "cropped_faces"
os.makedirs(SAVE_DIR, exist_ok=True)


def get_quality_score(image):
    """Calculates a quality score based on sharpness (Laplacian) and exposure."""
    sharpness = cv2.Laplacian(image, cv2.CV_64F).var()
    yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    brightness = np.mean(yuv[:, :, 0])
    exposure_score = 255 - abs(brightness - 128)
    return (sharpness * exposure_score), brightness


# 1. Initialize YOLO for Person Detection/Tracking
model = YOLO("yolo11n.pt")

# 2. Initialize MediaPipe for Face Detection
mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.6)

best_shots = {}  # {track_id: {'score': float, 'crop': image}}

cap = cv2.VideoCapture("../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # Step 1: Detect and Track Persons
    results = model.track(frame, persist=True, classes=[0], verbose=False)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.int().cpu().tolist()
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, tid in zip(boxes, track_ids):
            x1, y1, x2, y2 = box
            person_roi = frame[max(0, y1) : y2, max(0, x1) : x2]
            if person_roi.size == 0:
                continue

            # Step 2: Detect Face inside Person ROI
            face_results = face_detector.process(cv2.cvtColor(person_roi, cv2.COLOR_BGR2RGB))

            if face_results.detections:
                # Pick the most confident face in the person ROI
                detection = max(face_results.detections, key=lambda d: d.score[0])
                fb = detection.location_data.relative_bounding_box
                ph, pw, _ = person_roi.shape

                fx, fy = int(fb.xmin * pw), int(fb.ymin * ph)
                fw, fh = int(fb.width * pw), int(fb.height * ph)

                face_crop = person_roi[max(0, fy) : fy + fh, max(0, fx) : fx + fw]
                if face_crop.size == 0:
                    continue

                # Step 3: Quality Check & One Best Shot Logic
                score, brightness = get_quality_score(face_crop)

                if THRESHOLD_MIN < brightness < THRESHOLD_MAX:
                    if tid not in best_shots or score > best_shots[tid]["score"]:
                        best_shots[tid] = {"score": score, "crop": face_crop.copy()}
                        # Optional: Auto-save intermediate best
                        cv2.imwrite(os.path.join(SAVE_DIR, f"person_{tid}.jpg"), face_crop)

    cv2.imshow("Multi-Stage Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

# cap.release()
# cv2.destroyAllWindows()

# Save final unique best faces
print(f"\n✅ Video processed. Found {len(best_shots)} unique persons.")
for tid, data in best_shots.items():
    filename = os.path.join(SAVE_DIR, f"person_{tid}_best.jpg")
    cv2.imwrite(filename, data["crop"])
    print(f"✨ Saved: {filename} (Brightness: {data['brightness']:.1f}, Quality Score: {data['score']:.1f})")


In [ ]:
import mediapipe as mp
import cv2
import numpy as np
import os
from scipy.spatial import distance as dist

# Configuration
SAVE_DIR = "cropped_faces"
os.makedirs(SAVE_DIR, exist_ok=True)
MIN_SHARPNESS = 100  # Adjust based on camera quality

# Initialize MediaPipe
mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.6)


def get_sharpness(image):
    return cv2.Laplacian(image, cv2.CV_64F).var()


def get_exposure_score(image):
    yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    avg_brightness = np.mean(yuv[:, :, 0])
    return 255 - abs(avg_brightness - 128)


# Basic Centroid Tracker Logic
next_id = 0
trackers = {}  # {id: centroid}
best_shots = {}  # {id: {score, crop, sharpness}}

cap = cv2.VideoCapture("../media_files/গোয়াল থেকে গরু চুরির দু_র্ধ_র্ষ দৃশ্য ধরা পড়লো সিসিটিভিতে 720p.mp4")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    ih, iw, _ = frame.shape
    results = face_detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    current_frame_faces = []

    if results.detections:
        for detection in results.detections:
            bbox = detection.location_data.relative_bounding_box
            x, y, w, h = int(bbox.xmin * iw), int(bbox.ymin * ih), int(bbox.width * iw), int(bbox.height * ih)
            cx, cy = x + w // 2, y + h // 2
            current_frame_faces.append({"box": (x, y, w, h), "centroid": (cx, cy)})

    # Match detected faces to existing IDs (Simple Centroid Tracking)
    if not trackers:
        for f in current_frame_faces:
            trackers[next_id] = f["centroid"]
            next_id += 1
    else:
        ids = list(trackers.keys())
        coords = list(trackers.values())

        for f in current_frame_faces:
            # Find closest existing ID
            dists = dist.cdist([f["centroid"]], coords)
            idx = np.argmin(dists)

            if dists[0][idx] < 100:  # Threshold for "same person"
                matched_id = ids[idx]
                trackers[matched_id] = f["centroid"]

                # Quality Assessment
                x, y, w, h = f["box"]
                face_roi = frame[max(0, y) : y + h, max(0, x) : x + w]
                if face_roi.size == 0:
                    continue

                sharpness = get_sharpness(face_roi)
                exposure = get_exposure_score(face_roi)
                # Combined Score: Sharpness * Exposure * Size
                total_score = sharpness * exposure * (w * h)

                if matched_id not in best_shots or total_score > best_shots[matched_id]["score"]:
                    # Crop with padding
                    pw, ph = int(w * 0.2), int(h * 0.2)
                    crop = frame[max(0, y - ph) : min(ih, y + h + ph), max(0, x - pw) : min(iw, x + w + pw)]
                    best_shots[matched_id] = {"score": total_score, "crop": crop.copy()}
            else:
                # New person detected
                trackers[next_id] = f["centroid"]
                next_id += 1

    cv2.imshow("Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
# Save logic remains same...
cv2.destroyAllWindows()